### This notebook demonstrate how to create a more interesting paleo-map using GPlately.

In [ ]:
import warnings
from pathlib import Path
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from gplately import  Raster, auxiliary
from gplately.plot.gmt_cpt import get_cmap_from_gmt_cpt

warnings.filterwarnings("ignore", category=UserWarning, module="gplately")

data_dir = Path("./gplately-example-data")
data_dir.mkdir(parents=True, exist_ok=True)
# download age grid CPT file from https://raw.githubusercontent.com/GPlates/gplately/refs/heads/master/tests-dir/unittest/create-age-grids-video/agegrid.cpt
cpt_file = data_dir / "agegrid.cpt"
if not cpt_file.is_file():
    import urllib.request

    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/GPlates/gplately/refs/heads/master/tests-dir/unittest/create-age-grids-video/agegrid.cpt",
        cpt_file,
    )

In [ ]:
# use the auxiliary function to create a PlotTopologies instance
gplot = auxiliary.get_gplot("Zahirovic2022", time=100)  # 100Ma
assert gplot.time is not None

# download the age grid at the reconstruction time
agegrid = Raster(
    data=gplot.plate_reconstruction.plate_model.get_raster("AgeGrids", int(gplot.time))  # type: ignore
)


fig = plt.figure(figsize=(8, 4))
ax1 = fig.add_subplot(111, projection=ccrs.Mollweide(190))

# plot something for fun
gplot.plot_continents(ax1, facecolor="0.8")
gplot.plot_coastlines(ax1, color="0.5")
gplot.plot_all_topological_sections(
    ax1,
    plot_subduction_teeth=True,
    other_kwargs={
        "color": "grey",
        "linewidth": 0.5,
    },
    ridge_kwargs={
        "color": "black",
        "linewidth": 0.7,
    },
    transform_kwargs={
        "color": "green",
        "linewidth": 0.7,
    },
    trench_kwargs={
        "color": "blue",
        "linewidth": 0.7,
    },
)
im = gplot.plot_grid(
    ax1, agegrid.data, cmap=get_cmap_from_gmt_cpt(cpt_file), vmin=0, vmax=200
)
gplot.plot_plate_motion_vectors(
    ax1, spacingX=10, spacingY=10, normalise=True, zorder=10, alpha=0.5
)
assert im is not None

fig.colorbar(im, orientation="horizontal", shrink=0.4, pad=0.05, label="Age (Ma)")
plt.title(f"{int(gplot.time)} Ma")


# save the map as a .png file
output_file = f"{data_dir}/03-PlotWithCartopy.png"
fig.savefig(output_file, dpi=120, bbox_inches="tight")  # transparent=True)
print(f"Done! The {output_file} has been saved successfully.")

plt.show()
plt.close(fig)